<a href="https://colab.research.google.com/github/ChanchalSaha48/nlp-learning-journey/blob/main/03_machine_learning/02_Logistic%20Regression/Logistic_Regression_Interpretablity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression Interpretability and Error Analysis

## Objective

In this notebook, I will investigate how a Logistic Regression model makes predictions for text data.

I will explore:

- Model weights
- Decision score
- Probability
- Prediction
- Feature contribution
- Positive and negative evidence

- False positive and False Negative examples

In [1]:
!pip install opendatasets


In [2]:
#================================
# Import required libraries
#=================================
import warnings
warnings.filterwarnings('ignore')

import opendatasets as od
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [3]:
od.download('https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews')

Skipping, found downloaded files in "./imdb-dataset-of-50k-movie-reviews" (use force=True to force download)


In [4]:
df=pd.read_csv('/content/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [5]:
def preprocessing(text):
  text=BeautifulSoup(text,'html.parser').get_text()
  text=text.lower()
  return text


In [ ]:
df['review']=df['review'].apply(preprocessing)

KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__del__'
Traceback (most recent call last):
  File "_zmq.py", line 160, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


In [6]:
# Separate input and target

x=df['review']
y=df['sentiment'].map({
    'negative':0,
    'positive':1
})

In [7]:
# Split the dataset

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

print('Training samples: ',x_train.shape)
print('Testing samples: ',x_test.shape)

Training samples:  (40000,)
Testing samples:  (10000,)


In [8]:
# Convert text to TF-IDF features

tfidf=TfidfVectorizer(max_features=20000)
x_train_tfidf=tfidf.fit_transform(x_train)
x_test_tfidf=tfidf.transform(x_test)

In [9]:
print("Traing shape: ",x_train_tfidf.shape)
print("Testing shape: ",x_test_tfidf.shape)

Traing shape:  (40000, 20000)
Testing shape:  (10000, 20000)


In [10]:
# Train Logistic Regression

model=LogisticRegression(max_iter=1000)

In [11]:
model.fit(x_train_tfidf,y_train)
print(" Model trained successfully")

 Model trained successfully


## Select one Review

In [12]:
review_index=0

review=x_test.iloc[review_index]
actual_label=y_test.iloc[review_index]

print("Review: ",review)
print("Actual label: ",actual_label)


Review:  Yes, MTV there really is a way to market Daria. What started as a clever teenage angst-"comment on everything that sucks and make the viewer feel better about their sucky teenage life" sitcom now mutated into a "how you should deal with your problems"-charade. I used to watch Daria all the time and loved it. Now, sitting here after watching the so called "movie" I can only wonder what the point of this all was. Daria tells us how to lead out life in college? Excuse me? didn't the point Daria made every episode that what you like to do is ok, as long as it is ok with yourself no matter what the rest of the sick sad world thinks of it? This entire thing reminded me of the scene in "Reality Bites" the movie channel shows the documentry for the first time.
Actual label:  0


## Conver review to TF-IDF

In [13]:
vector=tfidf.transform([review])

print("Vector shape: ",vector.shape)

Vector shape:  (1, 20000)


(1,2000) it means
1 -> review
20000-> features

## Prediction

In [14]:
prediction=model.predict(vector)[0]
print(prediction)

0


# Probability

In [15]:
proba=model.predict_proba(vector)[0]
print(f"Negative probability: {proba[0]*100:.2f}%")
print(f"Positive probability: {proba[1]*100:.2f}%")

Negative probability: 53.18%
Positive probability: 46.82%


## Decision Score

In [16]:
decision_score=model.decision_function(vector)[0]
print("Decision score, z = ",decision_score)

Decision score, z =  -0.12719858576401089


# Verify Sigmoid Manually

In [17]:
# calculate probability manually using sigmoid

manual_probability= 1/(1+np.exp(-decision_score))
print("Manual positive probability: ",manual_probability)
print("Model positive probability: ",proba[1])

Manual positive probability:  0.4682431594485107
Model positive probability:  0.4682431594485107


## Compare Everything

In [18]:
# Predict summary

print("Actual label: ",actual_label)
print("Prediction: ",prediction)

print("Negative Probability: ",proba[0])
print('Positive Probability: ',proba[1])

print("Decision score: ",decision_score)

Actual label:  0
Prediction:  0
Negative Probability:  0.5317568405514893
Positive Probability:  0.4682431594485107
Decision score:  -0.12719858576401089


## Feature Names

In [19]:
feature_names= tfidf.get_feature_names_out()
print("Number of features: ",len(feature_names))

Number of features:  20000


## Which words present in Review

In [20]:
indices=vector.nonzero()[1]  # 0-->row no 1-->features index(col) no.
print("Number of non-zero features: ",len(indices))

Number of non-zero features:  95


## Create Empty list to store Contribution

In [21]:
contributions=[]


In [22]:
# Calculate each feature's contribution

for index in indices:
  # get the word corresponding to this feature index
  word=feature_names[index]

  # Get the TF-IDF value of this word
  tfidf_value=vector[0,index]

  # Get the Logistic Regression weight of this word
  weight=model.coef_[0][index]

  #calculate the contribution
  contribution= tfidf_value*weight

  #store the information
  contributions.append([word,tfidf_value,weight,contribution])


In [23]:
# Create a DataFrame from the contributions

explanation=pd.DataFrame(contributions,
                         columns=["word",'tfidf','weight','contribution'])

explanation.head(20)

,word,tfidf,weight,contribution
0,about,0.034452,-0.254456,-0.008767
1,after,0.045341,-0.283482,-0.012853
2,all,0.060475,-0.168335,-0.010180
3,and,0.037844,2.809427,0.106319
4,angst,0.124568,-0.156075,-0.019442
5,as,0.079317,0.903281,0.071646
6,better,0.049167,-1.675969,-0.082402
7,bites,0.136649,-0.129297,-0.017668
8,called,0.072268,-0.978756,-0.070733
9,can,0.035948,0.310909,0.011176


## Position Contribution


In [24]:
positive_contribution=explanation[explanation['contribution']>0].sort_values(
    by='contribution',
    ascending=False
 )

positive_contribution.head(20)

,word,tfidf,weight,contribution
37,loved,0.072621,4.724576,0.343102
15,daria,0.619161,0.472334,0.292451
34,life,0.099068,2.079492,0.206011
32,it,0.061258,3.310168,0.202775
92,you,0.058578,2.838535,0.166276
62,shows,0.065141,2.338112,0.152307
90,world,0.058179,2.163325,0.125859
80,us,0.058935,1.967326,0.115944
14,comment,0.086795,1.281745,0.111250
55,reality,0.080535,1.326150,0.106802


## Negative Contribution

In [25]:
negative_contribution=explanation[explanation['contribution']<0].sort_values(
    by='contribution',
    ascending=True
)
negative_contribution.head(10)

,word,tfidf,weight,contribution
49,ok,0.158193,-1.815839,-0.287252
68,sucks,0.101477,-2.727871,-0.276815
53,point,0.120158,-2.039477,-0.245059
23,excuse,0.094644,-1.797312,-0.170104
46,no,0.038867,-4.019971,-0.156246
89,wonder,0.077574,-1.812638,-0.140614
58,rest,0.068597,-2.013318,-0.138108
51,only,0.038713,-3.263042,-0.126323
63,sick,0.092916,-1.172318,-0.108927
17,didn,0.054404,-1.979979,-0.107719


## Total Contribution

In [26]:
total_positive_contribution=positive_contribution['contribution'].sum()
total_negative_contribution=negative_contribution['contribution'].sum()

print("Total Negative contribution: ",total_negative_contribution)
print("Total positive contribution: ",total_positive_contribution)

Total Negative contribution:  -3.616339213468458
Total positive contribution:  3.460654353405546


In [27]:
# Get model bias

bias=model.intercept_[0]
print("Bias: ",bias)

Bias:  0.028486274298900608


Calculate Decision Score Manually

In [28]:
total_contribution= total_negative_contribution+ total_positive_contribution
manual_score=total_contribution+bias
print("Manual Decision Score: ",manual_score)
print("Model decision score: ",decision_score)

Manual Decision Score:  -0.12719858576401133
Model decision score:  -0.12719858576401089


## Manual Probability

In [29]:
manual_probability=1/(1+np.exp(-manual_score))
print("Manual probability: ",manual_probability)
print("Model Probability: ",proba[1])

Manual probability:  0.4682431594485106
Model Probability:  0.4682431594485107


Manual Prediction

In [30]:
if manual_probability>0.5:
  print(1)
else:
  print(0)

0


## Error Analysis

In [33]:
y_pred=model.predict(x_test_tfidf)
error=x_test[y_test!=y_pred]

error_actual=y_test[y_test!=y_pred]
error_pred=y_pred[y_test!=y_pred]

error_df=pd.DataFrame({
    'review':error,
    'actual':error_actual,
    'predicted':error_pred
})

In [34]:
error_df.head(20)

,review,actual,predicted
39791,The story of the bride fair is an amusing and ...,0,1
40714,Little Quentin seems to have mastered the art ...,0,1
4142,The movie 'Gung Ho!': The Story of Carlson's M...,0,1
11840,I had known Brad Linaweaver at Florida State U...,0,1
48388,In order to enjoy 'Fur - An imaginary portrait...,0,1
20169,I am quite a fan of novelist/screenwriter Mich...,0,1
26800,Tenshu is imprisoned and sentenced to death. W...,1,0
46536,I just can't believe some of the comments on t...,1,0
39806,1. I've seen Branaghs Hamlet: Branagh is too o...,1,0
45621,"Ladies and gentlemen, we've really got ourselv...",1,0


## False Positive
actual->0
pred-->1

In [35]:
false_positive=error_df[(error_df['actual']==0) & (error_df['predicted']==1)]
print("Number of False Positive: ",false_positive.shape)
false_positive.head(10)

Number of False Positive:  (523, 3)


,review,actual,predicted
39791,The story of the bride fair is an amusing and ...,0,1
40714,Little Quentin seems to have mastered the art ...,0,1
4142,The movie 'Gung Ho!': The Story of Carlson's M...,0,1
11840,I had known Brad Linaweaver at Florida State U...,0,1
48388,In order to enjoy 'Fur - An imaginary portrait...,0,1
20169,I am quite a fan of novelist/screenwriter Mich...,0,1
28165,I appeared as an extra and was on location as ...,0,1
32867,In my opinion of this movie the entire video p...,0,1
32121,"If you are one of the people who finds ""Accord...",0,1
23424,Hollywood had a long love affair with bogus Ar...,0,1


## False Negative
actual->1 pred-->0

In [36]:
false_negative=error_df[(error_df['actual']==1)&(error_df['predicted']==0)]
print("Number of false Negative: ",false_negative.shape)
false_negative.head(20)

Number of false Negative:  (471, 3)


,review,actual,predicted
26800,Tenshu is imprisoned and sentenced to death. W...,1,0
46536,I just can't believe some of the comments on t...,1,0
39806,1. I've seen Branaghs Hamlet: Branagh is too o...,1,0
45621,"Ladies and gentlemen, we've really got ourselv...",1,0
1396,Citizen Kane....The Godfather Part II....D'Urv...,1,0
5000,"Not a movie for everyone, but this movie is in...",1,0
35526,Red Eye is a good little thriller to watch on ...,1,0
48433,"I think the film is educational. However, it f...",1,0
22201,"There is this private campground in Plymouth, ...",1,0
34541,This movie is a great way for the series to fi...,1,0


# Find out the reason for error




In [37]:
fn_review=false_negative['review'].iloc[0]
print(fn_review)

Tenshu is imprisoned and sentenced to death. When he survives electrocution the government officials give him a choice to either be electrocute at a greater degree or agree to some experiments. He chooses the experimentation and is placed in a large metallic cell with a bad ass criminal who also survived the electrocution. They can have whatever the want in the room (within reason), but they can't leave. after a few days there meals are cut down to one per day and the room temp is set up too 100. After some more alarms are sounded at intervals so they can't sleep. One day a 'witch' come into their cell (albeit a glassed off portion) What happens next I'll let you find out. I may be in the minority here but I liked the build up, it was intriguing to me. Now if the payoff was half as good as the build up was I would have rated this so much higher.<br /><br />My Grade: C+ <br /><br />Media Blaster's 2 DVD set Extras: Disc 1) Director's Cut; Trailers for "Versus", "Aragami", "Attack the Ga

In [38]:
fn_vector=tfidf.transform([fn_review])


In [39]:
fn_prediction=model.predict(fn_vector)[0]
fn_probability=model.predict_proba(fn_vector)[0][1]

print("Predcition: ",fn_prediction)
print("Positive Probability: ",fn_probability)


Predcition:  0
Positive Probability:  0.31778303590663415


In [44]:
fn_indices=fn_vector.nonzero()[1]
fn_contributions=[]

for index in fn_indices:

  word=feature_names[index]
  tfidf=fn_vector[0,index]
  weight=model.coef_[0][index]
  fn_contributions.append([word,tfidf,weight,(tfidf*weight)])



In [45]:
fn_df=pd.DataFrame(
    fn_contributions,
    columns=['word','tfidf','weight','contribution']
)

In [49]:
print("Positive effect for those words: ")
pos_con=fn_df[fn_df['contribution']>0].sort_values(by='contribution',ascending=False)
pos_con.head(20)

Positive effect for those words: 


,word,tfidf,weight,contribution
5,and,0.102242,2.809427,0.287240
34,dvd,0.058247,3.467632,0.201981
68,liked,0.065303,2.921071,0.190756
18,cell,0.206239,0.740694,0.152760
4,also,0.038868,2.928313,0.113817
32,disc,0.217010,0.493132,0.107015
71,may,0.053389,1.850198,0.098780
44,good,0.032358,3.009081,0.097368
26,day,0.113695,0.706638,0.080342
62,is,0.054962,1.403150,0.077120


In [48]:
print("Negative effect for those words: ")
neg_con=fn_df[fn_df['contribution']<0].sort_values(by='contribution',ascending=True)
neg_con.head(20)

Negative effect for those words: 


,word,tfidf,weight,contribution
11,bad,0.040186,-8.902451,-0.357753
46,grade,0.083221,-2.104597,-0.175146
48,half,0.060287,-2.828934,-0.170548
25,cut,0.214698,-0.679128,-0.145807
86,original,0.056275,-2.512850,-0.141411
95,reason,0.058279,-2.338449,-0.136282
31,director,0.048753,-2.648872,-0.129140
99,sleep,0.089830,-1.338847,-0.120268
119,was,0.071025,-1.678623,-0.119224
35,either,0.061039,-1.886966,-0.115179


In [62]:
pc=pos_con['contribution'].sum()
print("Toal positive contribution : ",pc)

Toal positive contribution :  3.141428006148275


In [63]:
nc=neg_con['contribution'].sum()
print("Total Negative contribution: ",nc)

Total Negative contribution:  -3.9338931439673175


In [64]:
b=model.intercept_[0]
print("Bias: ",b)


Bias:  0.028486274298900608


In [67]:
score=pc+nc+b
print(score)
manual_pred=1/(1+np.exp(-score))
print("Negative" if manual_pred<0.5  else "positive")

-0.7639788635201418
Negative
